## 0. Setup (run first, every session) - and how your progress is protected
**Colab:** checkpoints/results are written *directly to your Google Drive* (symlinks), so a power or internet cut loses at most one checkpoint interval (~300 updates). After reconnecting: re-run section 0, then re-run the same training cell - it resumes automatically. Drive's free quota is 15 GB: one full-FT run is ~12 GB, so use the cleanup cell after each full-FT run.

**Kaggle:** an interactive session's disk is wiped when it ends. Protect yourself with **Save Version -> Save & Run All (Commit)**: it runs on Kaggle's servers even if your internet/power dies, and the output (checkpoints + results, up to 20 GB) is kept. To continue later: in the new notebook use *Add Input -> Notebook Output Files* and pick the previous version; section 0 copies its checkpoints back and training resumes. Do one method per commit run (must finish within the 9 h limit). Settings: Accelerator = GPU, Internet = On.

4. **Not yet run end to end.** Offline steps (3.1-3.3, 3.5) are CPU-only and can run locally; only the live generation (3.4) needs a GPU.
5. Provide the supplied rollout data under `data/ATML-PA3-Data/` (download with the id from LMS, cell 3.0).

In [ ]:
import os, sys, subprocess, shutil, glob
REPO_URL = "https://github.com/ZainFarhan-Work/ATML.git"
BRANCH   = "main"
ON_KAGGLE = os.path.exists("/kaggle")
ROOT = "/kaggle/working" if ON_KAGGLE else "/content"

if not os.path.exists(f"{ROOT}/ATML"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, f"{ROOT}/ATML"], check=True)
else:
    subprocess.run(["git", "-C", f"{ROOT}/ATML", "pull"], check=False)
os.chdir(f"{ROOT}/ATML/PA 3")
print(os.getcwd())

PERSIST = ["checkpoints", "results", "cached", "splits"]
if not ON_KAGGLE:
    # Colab: keep run state on Drive so it survives disconnects
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/atml_pa3"
    for d in PERSIST:
        os.makedirs(f"{DRIVE}/{d}", exist_ok=True)
        if os.path.isdir(d) and not os.path.islink(d):
            shutil.copytree(d, f"{DRIVE}/{d}", dirs_exist_ok=True); shutil.rmtree(d)
        if not os.path.islink(d):
            os.symlink(f"{DRIVE}/{d}", d)
else:
    # Kaggle: restore from earlier committed runs attached as inputs
    for src_root in glob.glob("/kaggle/input/*/ATML/PA 3") + glob.glob("/kaggle/input/*/*/ATML/PA 3"):
        for d in PERSIST:
            if os.path.isdir(f"{src_root}/{d}"):
                shutil.copytree(f"{src_root}/{d}", d, dirs_exist_ok=True)
                print("restored", d, "from", src_root)

def sync_now():
    # Colab: nothing to do (already on Drive). Kaggle: output is saved when the committed run finishes.
    print("progress is stored in:", "Google Drive" if not ON_KAGGLE else "/kaggle/working (kept by Save & Run All)")

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m scripts.check_environment

## 3.0 Data

In [ ]:
# !python -m scripts.download_assets --task3-repo <hf-dataset-id-from-LMS>
!python -m scripts.inspect_task3_data

## 3.1 Repeated sampling, scoring and aggregation (Steps 1-2)

In [ ]:
for ds in ["math", "mathvista"]:
    !python -m task3_ttc.offline --dataset {ds} --workers 4
sync_now()

## 3.2 What the signals capture (Step 3)

In [ ]:
for ds in ["math", "mathvista"]:
    !python -m task3_ttc.signals --dataset {ds}

## 3.3 Parallel vs sequential, live generation (Step 4) - GPU

In [ ]:
!python -m task3_ttc.live
sync_now()

## 3.4 Adaptive sampling (Step 5)

In [ ]:
for ds in ["math", "mathvista"]:
    !python -m task3_ttc.adaptive --dataset {ds}
sync_now()

## 3.5 Compute allocation and failure cases (Step 6)

In [ ]:
!python -m task3_ttc.analyze
sync_now()